<a id="inizio"></a>
# 11 · Capstone: il carico del Nord e la temperatura · Soluzioni

⏱ ~110 min &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/load_total_north_hourly_2025.xlsx`, `../Dati/fallback/meteo_milano_2024_2025.json`

**In questo notebook impariamo a**

- caricare, unire e pulire una serie temporale reale, con i salti dell'ora legale
- costruire la serie giornaliera e i profili medi per ora e per giorno della settimana
- unire il carico con la temperatura di Open-Meteo e leggere la relazione tra i due

I dati vengono dal [Download Center di Terna](https://dati.terna.it/en/download-center): il **carico
elettrico della zona Nord** negli anni 2024 e 2025. Nei dataset Terna il carico è la potenza richiesta
al sistema elettrico in ciascun intervallo temporale, una grandezza centrale per la pianificazione e la
gestione della rete di trasmissione.

Accanto al carico integriamo la **temperatura** di Milano, presa tramite API da
[Open-Meteo](https://open-meteo.com). L'obiettivo è descrivere come il carico del Nord dipende dall'ora
del giorno, dal giorno della settimana e dalla temperatura.

Per i titoli e le etichette dei grafici si può usare Copilot; il codice degli step lo scriviamo noi.

<a id="sez-1"></a>
## 1. Caricamento dei dati e unione

<a id="es-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 1 · Caricamento dei dati e unione**

In questo step costruiamo una singola serie storica continua a partire dai dati grezzi.

**Cosa fare.**

1. Leggi i due file Excel presenti nella cartella `../Dati/` in `df_2024` e `df_2025`:
   `load_total_north_hourly_2024.xlsx` e `load_total_north_hourly_2025.xlsx`.
2. Unisci i due dataset **per righe** in `df`, mantenendo lo stesso schema di colonne, in modo da
   ottenere una serie unica che copra l'intero periodo 2024-2025.
3. Controlla dimensione del dataset, prima e ultima data, quante righe ci sono in media per giorno
   (in `righe_giorno`) e i valori mancanti per colonna. Il file si chiama `hourly`: la frequenza è
   oraria?

Risultato atteso: 70.176 righe e 4 colonne, nessun valore mancante.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 1**

</div>

In [ ]:
import pandas as pd

df_2024 = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
df_2025 = pd.read_excel("../Dati/load_total_north_hourly_2025.xlsx")

df = pd.concat([df_2024, df_2025], ignore_index=True)

print(df.shape)
print(df["Date"].min(), df["Date"].max())
righe_giorno = len(df) / df["Date"].dt.date.nunique()
print(f"{righe_giorno:.0f} righe al giorno")
df.isna().sum()

In [ ]:
from corso import verifica
verifica("Step 1")

96 righe al giorno: i dati sono quartorari, un valore ogni 15 minuti, anche se il nome del file dice
`hourly`.

<a id="sez-2"></a>
## 2. Colonna temporale e prima visualizzazione

<a id="es-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 2 · Preparazione della colonna temporale e prima visualizzazione**

In questo step rendiamo la colonna data utilizzabile come variabile temporale e facciamo una prima ispezione visiva della serie.

**Cosa fare.**

1. Controlla con `df.dtypes` che `Date` sia già `datetime`; se fosse testo, `pd.to_datetime` la
   converte.
2. Ordina il dataset per data (i file partono dall'ultimo quarto d'ora dell'anno) e reimposta
   l'indice per avere un ordinamento pulito.
3. Importa Plotly e visualizza **entrambe** le serie nel tempo: `Total Load [MW]` e
   `Forecast Total Load [MW]`.
4. Dopo aver verificato che le due curve siano coerenti, elimina le colonne che non useremo:
   `Forecast Total Load [MW]` e `Bidding Zone`.

Al termine di questo step deve rimanere una serie storica con una colonna temporale in formato
`datetime` e una sola colonna di valori, `Total Load [MW]`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 2**

</div>

In [ ]:
import plotly.express as px

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

fig = px.line(df, x="Date", y=["Total Load [MW]", "Forecast Total Load [MW]"],
              title="Carico e previsione di Terna, zona Nord",
              labels={"value": "carico [MW]", "variable": "serie"})
fig.show()

df = df.drop(columns=["Forecast Total Load [MW]", "Bidding Zone"])
df.head()

In [ ]:
from corso import verifica
verifica("Step 2")

<a id="sez-3"></a>
## 3. Ora legale: duplicati e buchi

Prima di aggregare controlliamo il passo tra un timestamp e il successivo. Due giorni all'anno qualcosa
non torna: l'ultima domenica di ottobre e l'ultima domenica di marzo, quando cambia l'ora.

Proviamo prima a dare alle date il fuso orario italiano e a lasciare a pandas il riconoscimento
dell'ora ripetuta. La cella dà errore apposta: leggiamo l'ultima riga.

In [ ]:
df["Date"].dt.tz_localize("Europe/Rome", ambiguous="infer")

`There are 4 dst switches when there should only be 1`: le due copie di ogni quarto d'ora ripetuto
sono intrecciate, e pandas vede quattro cambi d'ora invece di uno. Per questa analisi basta l'ora
locale senza fuso: togliamo i doppioni con `drop_duplicates`.

<a id="es-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 3 · Pulizia dei duplicati dell'ora legale**

In questo step lasciamo una riga per timestamp e contiamo i quarti d'ora che mancano.

**Cosa fare.**

1. In `passi` metti la differenza tra ogni timestamp e il precedente (`diff()`) e conta quante volte
   compare ogni valore con `value_counts()`. Qual è il passo normale? Cosa sono gli altri due valori?
2. In `duplicati` metti le righe con un timestamp ripetuto, tutte le copie
   (`duplicated(keep=False)`). Che giorni sono? Che ore?
3. Ordina per `Date`, togli i duplicati con `drop_duplicates(subset="Date")` e reimposta l'indice.
4. In `buchi` metti le righe che arrivano dopo un salto più lungo di 15 minuti. Che giorni sono?

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 3**

Teniamo la prima copia: le due differiscono di qualche centinaio di MW su circa 11.000. Il buco di marzo resta: quei quarti d'ora non ci sono stati.

</div>

In [ ]:
passi = df["Date"].diff()
print(passi.value_counts())

duplicati = df[df["Date"].duplicated(keep=False)]
print(duplicati)

df = df.sort_values("Date").drop_duplicates(subset="Date").reset_index(drop=True)

buchi = df[df["Date"].diff() > pd.Timedelta("15min")]
buchi

In [ ]:
from corso import verifica
verifica("Step 3")

<a id="sez-4"></a>
## 4. Aggregazioni temporali e profili medi

Useremo `Date` come colonna temporale e `Total Load [MW]` come variabile di interesse.

<a id="es-4"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 4 · Aggregazioni temporali e profili medi**

In questo step analizziamo la serie quartoraria cambiando punto di vista sul tempo. Costruiamo una
serie giornaliera e osserviamo i pattern medi legati all'ora del giorno e al giorno della settimana.

**Cosa fare.**

**Serie giornaliera (somma)**

1. Parti dal DataFrame quartorario `df`.
2. Imposta la colonna `Date` come indice temporale in `df_temp`.
3. Aggrega per giorno usando `resample("D")` e calcola la **somma** del carico.
4. Riporta `Date` come colonna e salva il risultato in `df_daily`. Aggiungi la colonna
   `Energia [MWh]`: la somma divisa per 4, perché un quarto d'ora a P MW vale P/4 MWh.
5. Visualizza la serie giornaliera con Plotly.

**Profilo medio giornaliero (media per ora del giorno)**

6. Torna alla serie quartoraria `df`: estrai l'**ora del giorno** dalla colonna `Date` e salvala in
   una nuova colonna `hour`.
7. Raggruppa per `hour` e calcola la **media** del carico in `profilo_orario`.
8. Visualizza il profilo medio giornaliero con Plotly.

**Profilo medio settimanale (media per giorno della settimana)**

9. Sempre partendo da `df`, estrai il **giorno della settimana** dalla colonna `Date` e salvalo in
   una nuova colonna `weekday`, con la convenzione `0 = lunedì, …, 6 = domenica`.
10. Raggruppa per `weekday` e calcola la **media** del carico in `profilo_settimanale`.
11. Visualizza il profilo medio settimanale con Plotly.

Guarda i tre grafici: in che periodi l'energia giornaliera scende? A che ore il carico è più alto,
e di quanto cala nel weekend?

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 4**

Somma e poi diviso 4, non media per 24: nei due giorni di marzo con un'ora in meno la media conterebbe un'ora che non c'è stata.

</div>

In [ ]:
df_temp = df.set_index("Date")
df_daily = df_temp["Total Load [MW]"].resample("D").sum()
df_daily = df_daily.reset_index()
df_daily["Energia [MWh]"] = df_daily["Total Load [MW]"] / 4
fig = px.line(df_daily, x="Date", y="Energia [MWh]", title="Energia giornaliera, zona Nord")
fig.show()

df["hour"] = df["Date"].dt.hour
profilo_orario = df.groupby("hour")["Total Load [MW]"].mean().reset_index()
fig = px.line(profilo_orario, x="hour", y="Total Load [MW]", markers=True,
              title="Profilo medio giornaliero del carico",
              labels={"hour": "ora del giorno", "Total Load [MW]": "carico medio [MW]"})
fig.update_xaxes(dtick=1)
fig.show()

df["weekday"] = df["Date"].dt.dayofweek
profilo_settimanale = df.groupby("weekday")["Total Load [MW]"].mean().reset_index()
profilo_settimanale["giorno"] = ["lunedì", "martedì", "mercoledì", "giovedì", "venerdì", "sabato", "domenica"]
fig = px.line(profilo_settimanale, x="giorno", y="Total Load [MW]", markers=True,
              title="Profilo medio settimanale del carico",
              labels={"giorno": "giorno della settimana", "Total Load [MW]": "carico medio [MW]"})
fig.show()

In [ ]:
from corso import verifica
verifica("Step 4")

<a id="sez-5"></a>
## 5. Recupero e integrazione della temperatura

Aggiungiamo la **temperatura a 2 metri** ricostruita per Milano. È una semplificazione: usiamo Milano
come punto "medio" per il Nord Italia. L'API
[Historical Weather](https://open-meteo.com/en/docs/historical-weather-api) di Open-Meteo non chiede
alcuna chiave e restituisce i dati orari su un intervallo di date.

<a id="es-5"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 5 · Recupero e integrazione della temperatura**

Ora dobbiamo integrare il dato di temperatura con il dato di carico. Per prima cosa va risolta la
differente granularità: la temperatura è oraria, il carico quartorario (da 1 ora a 15 minuti).

**Cosa fare.**

1. Chiedi all'archivio storico di Open-Meteo la temperatura oraria di Milano (latitudine 45.4642,
   longitudine 9.19) dal primo all'ultimo giorno del carico. URL
   `https://archive-api.open-meteo.com/v1/archive`; parametri `latitude`, `longitude`, `start_date`
   ed `end_date` come testo `AAAA-MM-GG`, `hourly="temperature_2m"`, `timezone="Europe/Rome"`.
   Salva in `orario` la chiave `hourly` della risposta.
2. Nella seconda cella trasforma `orario` in `meteo` con `pd.DataFrame`.
3. Converti `time` in datetime, rinomina le colonne in `Date` e `Temperature_C`, imposta `Date` come
   indice e ordinalo con `sort_index()`.
4. Fai il resample a quartorario per poter integrare con il carico (`resample("15min")` +
   `interpolate(method="linear")`), riporta `Date` come colonna e salva il risultato in `meteo_15`.
5. Fai il merge con il DataFrame del carico `df`, tenendo tutte le sue righe, e salva il risultato in
   `df_full`.
6. Mancano i dati dell'ultima ora del 2025: riempi la temperatura con `ffill()`.
7. Visualizza la temperatura con `px.line()`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 5**

L'interpolazione lineare stima i tre valori tra un'ora e la successiva: non è una misura. `how="left"` tiene tutte le righe del carico; le ultime tre del 2025, dopo le 23:00, restano senza temperatura e le riempie `ffill`.

</div>

In [ ]:
import requests

start_date = df["Date"].min().strftime("%Y-%m-%d")
end_date = df["Date"].max().strftime("%Y-%m-%d")
url = "https://archive-api.open-meteo.com/v1/archive"
params = {"latitude": 45.4642, "longitude": 9.19, "start_date": start_date, "end_date": end_date,
          "hourly": "temperature_2m", "timezone": "Europe/Rome"}
response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
orario = response.json()["hourly"]

Se la rete non risponde, la cella qui sotto carica da `../Dati/fallback/` temperature di esempio
(non misurate); se l'API ha risposto, saltala.

In [ ]:
import json

with open("../Dati/fallback/meteo_milano_2024_2025.json", encoding="utf-8") as f:
    orario = json.load(f)["hourly"]
len(orario["time"])

In [ ]:
meteo = pd.DataFrame(orario)
meteo["time"] = pd.to_datetime(meteo["time"])
meteo = meteo.rename(columns={"time": "Date", "temperature_2m": "Temperature_C"})
meteo = meteo.set_index("Date").sort_index()

meteo_15 = meteo.resample("15min").interpolate(method="linear")
meteo_15 = meteo_15.reset_index()

df_full = pd.merge(df, meteo_15, on="Date", how="left")
print(df_full["Temperature_C"].isna().sum(), "temperature mancanti dopo il merge")
df_full["Temperature_C"] = df_full["Temperature_C"].ffill()

fig = px.line(df_full, x="Date", y="Temperature_C", title="Temperatura a Milano",
              labels={"Temperature_C": "temperatura [°C]"})
fig.show()

In [ ]:
from corso import verifica
verifica("Step 5")

<a id="sez-6"></a>
## 6. Carico e temperatura

<a id="es-6"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 6 · Carico e temperatura**

Un punto per giorno: temperatura media sulle x, carico medio sulle y, un colore per mese. Dalla forma della nuvola di punti si legge come il carico dipende dalla temperatura.

**Cosa fare.**

1. Porta `df_full` a medie giornaliere in `giornaliero`: `Date` come indice, le colonne
   `Total Load [MW]` e `Temperature_C`, `resample("D").mean()`, poi `reset_index()`.
2. Aggiungi la colonna `mese` con il nome del mese (`dt.month_name()`): così lo scatter ha un colore
   per mese invece di una scala continua.
3. Disegna lo scatter con `px.scatter`: temperatura sulle x, carico sulle y, `color="mese"`. Che
   forma ha? A che temperatura il carico è più basso?
4. Completa le cinque frasi della cella dopo la verifica, con i numeri e i grafici degli step.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione Step 6**

Le medie giornaliere tolgono il ciclo dell'ora e del giorno: la relazione con la temperatura si vede giorno per giorno.

</div>

In [ ]:
df_temp = df_full.set_index("Date")
giornaliero = df_temp[["Total Load [MW]", "Temperature_C"]].resample("D").mean()
giornaliero = giornaliero.reset_index()
giornaliero["mese"] = giornaliero["Date"].dt.month_name()

fig = px.scatter(giornaliero, x="Temperature_C", y="Total Load [MW]", color="mese", hover_data=["Date"],
                 title="Carico medio giornaliero e temperatura media, zona Nord 2024-2025",
                 labels={"Temperature_C": "temperatura media [°C]", "Total Load [MW]": "carico medio [MW]"})
fig.show()

In [ ]:
from corso import verifica
verifica("Step 6")

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Le cinque frasi**

1. I dati sono quartorari, non orari: 70.168 righe pulite, 8 doppioni di ottobre tolti (4 per anno),
   8 quarti d'ora di marzo mancanti, che restano un buco.
2. L'energia giornaliera va da circa 280.000 MWh (1° gennaio 2024) a circa 645.000 MWh (luglio); i
   giorni più leggeri sono le feste, le domeniche e agosto.
3. Nella giornata media il carico è minimo alle 3 (circa 14.300 MW) e massimo tra le 9 e le 11
   (circa 23.500 MW), con un secondo rialzo verso le 19.
4. La domenica il carico medio sta intorno a 15.300 MW, un giorno feriale tra 20.400 e 21.900 MW.
5. Il carico è più basso nelle giornate miti, intorno ai 15-20 °C, e sale con il freddo e con il caldo;
   al Nord pesa di più il caldo, con i massimi a luglio. I valori esatti dipendono dalla fonte della
   temperatura.

</div>

---

**Fine del notebook 11.** Precedente: [10 · Agenti per il coding](10_Agenti_per_il_coding.ipynb)

<sub>Blocco 4 · Agenti per il coding e caso d'uso end-to-end · Giornata 2 · Aula Avanzata</sub>